# SentinelAI Quantum AI Hackathon Demo

A reproducible, local four-qubit demonstration of behavioural feature encoding, fidelity-kernel similarity, VQC inference, and an Isolation Forest comparison. Quantum outputs are experimental and never alter SentinelAI's production risk score.

In [ ]:
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

from sentinel_ai.models.evaluation import prepare_evaluation_dataset
from sentinel_ai.models import IsolationForestDetector
from sentinel_ai.quantum import (
    FEATURE_QUBIT_MAPPING, build_quantum_circuit, extract_quantum_features,
    normalize_quantum_features, quantum_kernel_similarity, run_vqc,
)
print('Ready — all execution is local and simulator-based.')

## 1. Generate representative SentinelAI data
The existing deterministic demo generator provides normal history and labelled suspicious scenarios. The split is chronological, and suspicious rows are evaluation-only.

In [ ]:
dataset = prepare_evaluation_dataset()
selected_event, selected_features = dataset.anomaly_test[0]
print(dataset.split)
print(f'Selected: {selected_event.event_id} / {selected_event.scenario} / {selected_event.employee_name}')

## 2. Select and normalize four behavioural features

| Qubit | Behavioural signal | SentinelAI sources |
|---|---|---|
| q0 | Login-time deviation | `login_hour_deviation` |
| q1 | Failed-login behaviour | `failed_login_count` |
| q2 | Download-volume anomaly | positive max of count/size deviations |
| q3 | Device/location anomaly | mean of device and location binary anomalies |

In [ ]:
raw = extract_quantum_features(selected_features)
normalized = normalize_quantum_features(raw)
mapping = pd.DataFrame([
    {'qubit': q, 'feature': label, 'source': source, 'raw': raw[key], 'normalized': normalized[key]}
    for key, label, q, source, _ in FEATURE_QUBIT_MAPPING
])
mapping

## 3. Encode and visualize the four-qubit circuit
Each normalized value becomes an `RY(πx)` and `RZ(πx/2)` rotation. A CX ring entangles q0→q1→q2→q3→q0.

In [ ]:
circuit = build_quantum_circuit(normalized)
print(circuit.draw(output='text', fold=-1))

## 4. Run the quantum feature map and fidelity kernel
The kernel is the exact statevector fidelity `|⟨φ(event)|φ(normal baseline)⟩|²`. The anomaly score is `1 - similarity`; neither value is an attack probability.

In [ ]:
normal_rows = [normalize_quantum_features(extract_quantum_features(features)) for _, features in dataset.training]
keys = list(normalized)
normal_baseline = {key: float(np.mean([row[key] for row in normal_rows])) for key in keys}
started = time.perf_counter()
kernel_similarity = quantum_kernel_similarity(normalized, normal_baseline)
kernel_ms = (time.perf_counter() - started) * 1000
print(f'Quantum similarity to normal baseline: {kernel_similarity:.4f}')
print(f'Quantum anomaly score: {1-kernel_similarity:.4f}')
print(f'Execution: {kernel_ms:.2f} ms on local noiseless statevector simulation')

## 5. Run VQC inference
The lightweight VQC has eight trainable RY angles and linear CX entanglement. It trains once on 12 deterministic representative rows and is cached for reuse. The model score is not calibrated confidence.

In [ ]:
vqc = run_vqc(normalized)
pd.Series(vqc)

## 6. Isolation Forest comparison
This is a descriptive side-by-side view. The classical model retains its production role; quantum analysis is advisory only.

In [ ]:
classical = IsolationForestDetector()
classical.fit([features for _, features in dataset.training])
classical_result = classical.score(selected_features)
comparison = pd.DataFrame([
    {'model': 'Isolation Forest', 'prediction': 'ANOMALOUS' if (classical_result.anomaly_percentile or 0) >= 95 else 'NORMAL', 'score': (classical_result.anomaly_percentile or 0)/100, 'meaning': 'normal-history nonconformity percentile', 'dimensions': 11, 'backend': 'scikit-learn CPU'},
    {'model': 'Quantum Fidelity Kernel', 'prediction': 'SIMILARITY', 'score': kernel_similarity, 'meaning': 'state fidelity to normal baseline', 'dimensions': 4, 'backend': 'Qiskit circuit + local exact statevector executor'},
    {'model': 'VQC', 'prediction': vqc['prediction'], 'score': vqc['modelScore'], 'meaning': 'uncalibrated circuit output', 'dimensions': 4, 'backend': vqc['executionBackend']},
])
comparison

## 7. Interpretation and limitations

- Experimental quantum analysis does **not** modify SentinelAI's production risk score, alerts, MITRE mappings, graph risk, severity, or containment.
- Execution uses a local noiseless Qiskit statevector simulator, not quantum hardware.
- The VQC uses a small deterministic representative training set for hackathon demonstration; no production accuracy is claimed.
- Fidelity similarity and the VQC output are not attack probabilities or calibrated confidence.
- The classical and quantum feature spaces differ, so their scores are not directly interchangeable.
- No quantum advantage, supremacy, or superiority over classical machine learning is claimed.